In [502]:
# 그래프관련 import 
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

# 프로그램 
import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier # 분류 클래스
from sklearn.neighbors import KNeighborsRegressor # 예측 클래스
from sklearn.linear_model import LinearRegression # 예측 클래스
from sklearn.preprocessing import PolynomialFeatures # 다항특성 자동생성 라이브러리
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error # 평균절대오차 오차범위 클래스
from sklearn.preprocessing import StandardScaler  # 정규화 클래스
from sklearn.model_selection import train_test_split

In [503]:
df = pd.read_csv('data/MultipleLinearRegressionData.csv')
df.head(5)

,hour,absent,place,score
0,0.5,3,Home,10
1,1.2,4,Library,8
2,1.8,2,Cafe,14
3,2.4,0,Cafe,26
4,2.6,2,Home,22


In [504]:
data = df.iloc[:,:-2].values
label = df.iloc[:,-1].values
# data
label

# df[['hour','absent']]
# data = df.iloc[:,:2].values
# data = df.iloc[:,:-2].values
# label = df.iloc[:,-1].values

array([ 10,   8,  14,  26,  22,  30,  42,  48,  38,  58,  60,  72,  62,
        68,  72,  58,  76,  86,  90, 100])

In [505]:
place

array([['Home'],
       ['Library'],
       ['Cafe'],
       ['Cafe'],
       ['Home'],
       ['Home'],
       ['Library'],
       ['Library'],
       ['Home'],
       ['Cafe'],
       ['Cafe'],
       ['Cafe'],
       ['Library'],
       ['Cafe'],
       ['Library'],
       ['Home'],
       ['Cafe'],
       ['Home'],
       ['Library'],
       ['Library']], dtype=object)

In [490]:
df['place'].values
# 원핫 인코딩 작업 
place = df['place'].to_numpy().reshape(-1,1) # 2차원배열로 가져움 
encoder = OneHotEncoder(sparse_output=False,drop='first')
result = encoder.fit_transform(place)
result


array([[1., 0.],
       [0., 1.],
       [0., 0.],
       [0., 0.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 1.],
       [0., 0.],
       [0., 1.],
       [1., 0.],
       [0., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.]])

In [491]:
# data + place 합치기
total_data = np.concatenate((data,result),axis=1)
total_data

array([[ 0.5,  3. ,  1. ,  0. ],
       [ 1.2,  4. ,  0. ,  1. ],
       [ 1.8,  2. ,  0. ,  0. ],
       [ 2.4,  0. ,  0. ,  0. ],
       [ 2.6,  2. ,  1. ,  0. ],
       [ 3.2,  0. ,  1. ,  0. ],
       [ 3.9,  0. ,  0. ,  1. ],
       [ 4.4,  0. ,  0. ,  1. ],
       [ 4.5,  5. ,  1. ,  0. ],
       [ 5. ,  1. ,  0. ,  0. ],
       [ 5.3,  2. ,  0. ,  0. ],
       [ 5.8,  0. ,  0. ,  0. ],
       [ 6. ,  3. ,  0. ,  1. ],
       [ 6.1,  1. ,  0. ,  0. ],
       [ 6.2,  1. ,  0. ,  1. ],
       [ 6.9,  4. ,  1. ,  0. ],
       [ 7.2,  2. ,  0. ,  0. ],
       [ 8.4,  1. ,  1. ,  0. ],
       [ 8.6,  1. ,  0. ,  1. ],
       [10. ,  0. ,  0. ,  1. ]])

In [492]:
# 학습데이터, 테스트데이터 분리 
train_data, test_data, train_label, test_label = train_test_split(data,label,random_state=42)
train_data
train_label

array([ 30,  72,  26,  90,  76,  68,  14,  58, 100,  22,  62,  48,  60,
        72,  42])

In [493]:
poly = PolynomialFeatures(degree=2,include_bias=False)
poly.fit(train_data)
train_poly = poly.transform(train_data)
test_poly = poly.transform(test_data)
test_poly

array([[ 0.5 ,  3.  ,  0.25,  1.5 ,  9.  ],
       [ 8.4 ,  1.  , 70.56,  8.4 ,  1.  ],
       [ 6.9 ,  4.  , 47.61, 27.6 , 16.  ],
       [ 1.2 ,  4.  ,  1.44,  4.8 , 16.  ],
       [ 4.5 ,  5.  , 20.25, 22.5 , 25.  ]])

In [494]:
lr = LinearRegression()
lr.fit(train_poly,train_label)

score = lr.score(train_poly,train_label)
score2 = lr.score(test_poly,test_label)

print('train 정확도 :', score)
print('test 정확도 :', score2) 

train 정확도 : 0.990849650473167
test 정확도 : 0.8653920861285799


In [495]:
ss = StandardScaler()
ss.fit(train_poly)

train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)


In [496]:
alpha_list = [0.001,0.01,0.1,1,10,100]
train_score = []
test_score = []
for alpha in alpha_list:
    ridge = Ridge(alpha=alpha)
    ridge.fit(train_scaled,train_label)
    score = ridge.score(train_poly,train_label)
    score2 = ridge.score(test_poly,test_label)
    train_score.append(score)
    test_score.append(score2)


In [497]:
train_score
test_score

[-354.7190244752088,
 -291.99467309497436,
 -38.92462862332427,
 -107.14367735477573,
 -148.80397537947218,
 -15.040148209983034]

In [498]:
print('train 정확도:', score)
print('test 정확도:', score2)

train 정확도: -20.44174831680054
test 정확도: -15.040148209983034


In [499]:
lasso = Lasso(alpha=10)
lasso.fit(train_scaled,train_label)
print('[최종 모델 성능]')
print('훈련 정확도:', ridge.score(train_scaled, train_label))
print('테스트 정확도:', ridge.score(test_scaled, test_label))
print(np.sum(lasso.coef_)) 


[최종 모델 성능]
훈련 정확도: 0.3885175831666926
테스트 정확도: 0.24534829215249243
14.016826833392782


In [500]:
# 9. 예측 및 오차 확인
result = ridge.predict(test_scaled)
mae = mean_absolute_error(test_label, result)
print('평균 절대 오차(MAE):', mae)

평균 절대 오차(MAE): 22.135645224448645


In [501]:
lr = LinearRegression()
lr.fit(train_poly,train_label)

score = lr.score(train_poly,train_label)
score2 = lr.score(test_poly,test_label)

print('train 정확도 :', score)
print('test 정확도 :', score2) 

train 정확도 : 0.990849650473167
test 정확도 : 0.8653920861285799
